## Lab 08

### Zadanie 1 - Regexy

Przejdź 15 zadań ze strony http://regexone.com/

### Zadanie 2 - Regexy

_Atoms_

Napisz program, który (używając wyrażeń regularnych) przefiltruje plik atoms.log tak by pozostały jedynie linie dot. bezproblemowego przebiegu eksperymentu.
czyli np. powinny zostać odrzucone takie linie jak
```
RUN 000039 COMPLETED. OUTPUT IN FILE yttrium.dat. 1 UNDERFLOW WARNING.
```
czy
```
RUN 000058 COMPLETED. OUTPUT IN FILE cerium.dat. ALGORITHM DID NOT CONVERGE AFTER 100000 ITERATIONS.
```
można jedynie założyć, że każda linijka pasuje do poniższego schematu
```
RUN [[NUMER Z DOKŁADNIE 6 CYFR]] COMPLETED. OUTPUT IN FILE [[NAZWA]].dat. [[COKOLWIEK]]
```

In [1]:
import re
from pathlib import Path
from typing import Iterable

SUCCESSFUL_ATOM_RUN = re.compile(
    r"^RUN (?P<run_id>\d{6}) COMPLETED\. OUTPUT IN FILE "
    r"(?P<filename>[^.\s]+)\.dat\.\s*$"
)


def is_successful_atom_run(line: str) -> bool:
    """Return True only for atom runs with no trailing warning or error."""
    return SUCCESSFUL_ATOM_RUN.fullmatch(line) is not None


def successful_atom_runs(lines: Iterable[str]) -> list[str]:
    """Return valid atom-run lines without their trailing newlines."""
    return [line.rstrip("\n") for line in lines if is_successful_atom_run(line)]


def successful_atom_runs_from_file(path: str | Path) -> list[str]:
    """Read and filter an atoms.log/atoms.txt file."""
    with open(path, encoding="utf-8") as atoms:
        return successful_atom_runs(atoms)


# Self-contained checks: no external log file required.
_atom_examples = [
    "RUN 000001 COMPLETED. OUTPUT IN FILE hydrogen.dat.\n",
    "RUN 000039 COMPLETED. OUTPUT IN FILE yttrium.dat. 1 UNDERFLOW WARNING.\n",
    "RUN 000058 COMPLETED. OUTPUT IN FILE cerium.dat. ALGORITHM DID NOT CONVERGE AFTER 100000 ITERATIONS.\n",
]
assert successful_atom_runs(_atom_examples) == [
    "RUN 000001 COMPLETED. OUTPUT IN FILE hydrogen.dat."
]
print("atoms checks: OK")

atoms checks: OK


### Zadanie 3 - Regexy

_Messages_

Napisz program który wypisze linie z logu systemowego "messages.txt" które zawierają informacje o nieprawidłowej nazwie użytkownika (invalid user)

In [2]:
INVALID_USER_LOG = re.compile(
    r"^(?P<month>[A-Z][a-z]{2})\s+(?P<day>\d{1,2})\s+"
    r"(?P<time>\d{2}:\d{2}):\d{2}\s+\S+\s+sshd(?:\[\d+\])?:\s+"
    r"Invalid user\s+(?P<user>\S+)\s+from\s+"
    r"(?P<address>(?:\d{1,3}\.){3}\d{1,3})(?:\s|$)"
)

POLISH_MONTHS = {
    "Jan": "stycznia", "Feb": "lutego", "Mar": "marca", "Apr": "kwietnia",
    "May": "maja", "Jun": "czerwca", "Jul": "lipca", "Aug": "sierpnia",
    "Sep": "września", "Oct": "października", "Nov": "listopada", "Dec": "grudnia",
}


def find_invalid_user_lines(lines: Iterable[str]) -> list[str]:
    """Return original sshd log lines reporting an invalid username."""
    return [line.rstrip("\n") for line in lines if INVALID_USER_LOG.match(line)]


def format_invalid_user_message(line: str) -> str | None:
    """Translate one invalid-user sshd entry, or return None when it does not match."""
    match = INVALID_USER_LOG.match(line)
    if match is None:
        return None

    data = match.groupdict()
    return (
        f'Niepoprawna nazwa użytkownika: "{data["user"]}", '
        f'połączenie z {data["address"]} nawiązano {int(data["day"])} '
        f'{POLISH_MONTHS[data["month"]]} o godz. {data["time"]}'
    )


def invalid_user_messages(lines: Iterable[str]) -> list[str]:
    """Return formatted messages for every matching sshd log entry."""
    return [
        message
        for line in lines
        if (message := format_invalid_user_message(line)) is not None
    ]


def invalid_user_messages_from_file(path: str | Path) -> list[str]:
    """Read messages.txt and return translated invalid-user entries."""
    with open(path, encoding="utf-8") as messages:
        return invalid_user_messages(messages)


# Self-contained checks: no external log file required.
_message_example = "Jun 29 20:18:40 noether sshd[5805]: Invalid user tester from 218.189.194.200\n"
assert find_invalid_user_lines([_message_example, "normal service message\n"]) == [
    _message_example.rstrip("\n")
]
assert format_invalid_user_message(_message_example) == (
    'Niepoprawna nazwa użytkownika: "tester", połączenie z 218.189.194.200 '
    "nawiązano 29 czerwca o godz. 20:18"
)
print("messages checks: OK")

messages checks: OK


Przerób program tak, żeby wyświetlał sformatowane komunikaty, tzn.
```
Jun 29 20:18:40 noether sshd[5805]: Invalid user tester from 218.189.194.200
```
powinno zostać przerobione na
```
Niepoprawna nazwa użytkownika: "tester", połaczenie z 218.189.194.200 nawiązano 29 czerwca o godz. 20:18
```

### Zadanie 4 - Collections
Uzupenij ponizsze funckje korzystajac z zaimporotwanych klas

In [3]:
from collections import defaultdict

def count_letters(word):
    """ Zwraca slownik gdzie kluczami sa litery, a wartosci ilosc ich wystepowania """
    output = defaultdict(int)
    for letter in word:
        output[letter] += 1
        
    return output


print(count_letters('aaaaa') == {'a': 5})
print(count_letters('abbccaaaa') == {'a': 5, 'b': 2, 'c': 2})
print(count_letters('abc') == {'a': 1, 'b': 1, 'c': 1})

True
True
True


In [4]:
def group_words_by_first_letter(text):
    """ Dzieli tekst po spacjach i zwraca slownik gdzie kluczami sa pierwsze litery, a wartosciami listy slow zaczynajacych sie na te litery"""
    output = defaultdict(list)
    for splitted in text.split():
        output[splitted[0]].append(splitted)
    return output

print(group_words_by_first_letter("ala ma kota") == {'a': ['ala'], 'm': ['ma'], 'k': ['kota']})
print(group_words_by_first_letter("ala ma kota ala ma psa") == {'a': ['ala', 'ala'], 'm': ['ma', 'ma'], 'k': ['kota'], 'p': ['psa']})   
print(group_words_by_first_letter("ala ma kota ale marysia ma konia") == {'a': ['ala', 'ale'], 'm': ['ma', 'marysia', 'ma'], 'k': ['kota', 'konia']})

True
True
True


### Zadanie 5 - Klasy
Uzupełnij brakujące metody poniższej klasy.

In [5]:
from collections.abc import Mapping
from types import MappingProxyType


class FrozenDictionary:
    """Immutable, hashable dictionary."""

    def __init__(self, dictionary: Mapping):
        """Create a frozen copy of a mapping."""
        self._data = MappingProxyType(dict(dictionary))

    def __hash__(self) -> int:
        """Return a hash independent of key insertion order."""
        return hash(frozenset(self._data.items()))

    def __eq__(self, other: object) -> bool:
        """Compare values with another frozen dictionary or mapping."""
        if isinstance(other, FrozenDictionary):
            return self._data == other._data
        if isinstance(other, Mapping):
            return self._data == other
        return NotImplemented

    def __repr__(self) -> str:
        return repr(dict(self._data))


# Self-contained checks.
dicts = [
    FrozenDictionary({'ala': 4}),
    FrozenDictionary({'ala': 1, 'jacek': 0}),
    FrozenDictionary({'ala': 4}),
    FrozenDictionary({'ala': 2}),
    FrozenDictionary({'jacek': 0, 'ala': 1}),
]

frozen_set = set(dicts)
assert dicts[0] == dicts[2]
assert dicts[0] != dicts[3]
assert len(frozen_set) == 3
assert all(item in frozen_set for item in dicts)
print(frozen_set)

{{'ala': 4}, {'ala': 2}, {'ala': 1, 'jacek': 0}}


### Zadanie 6 - Bag of Words
Napisz klasę "Bag of words", która będzie "workiem słów" z zadanego dokumentu. Oznacza to, że ma w sobie przechowywać informacje o tym ile razy każde słowo wystąpiło w dokumencie. Dodatkowo ma udostępniać następującą funkcjonalność:
* Można utworzyć go na dwa sposoby:
```
bow = BagOfWords("ala ma kota ala ma ala")
bow = BagOfWords(open("plik.txt"))
```
* Możliwość wyświetlania go po prostu pisząc print(bow) (kolejność taka sama jak przy przeglądaniu forem)
```
ala:3, ma:2, kota:1
```
* Sprawdzanie czy jakieś słowo należy do naszego worka pisząc
```
if 'ala' in bow:
    ...
```
* Przeglądanie słów w worku od najczęściej do najrzadziej występującego, czyli
```
for word in bow:
  print(word)
```
    powinniśmy zobaczyć
```
ala
ma
kota
```
* Możliwość dodawania dwóch worków, pisząc
```
bow1 = BagOfWords("ala ma kota ala ma ala")
bow2 = BagOfWords("tomek tez ma kota")
bow3 = bow1 + bow2
print('tomek' in bow1) # False
print('tomek' in bow3) # True
print('ala' in bow3) # True
print(bow3) # ala:3, ma:3, kota:2, tez:1, tomek:1
    ```
* Odczytywanie częstości wystąpień słów poprzez nawiasy kwadratowe
```
print(bow1["ala"]) # 3
print(bow3["ala"]) # 3
```
* Aktualizację cześtości wystąpień
```
bow3['tomek'] = 10
for el in bow3:
    print el
```
    powinno wyświetlić najpierw `tomek`

In [6]:
from collections import Counter


class BagOfWords:
    """Frequency bag of normalized words from text or a readable text stream."""

    _WORD = re.compile(r"\w+(?:['-]\w+)*", re.UNICODE)

    def __init__(self, document: str | object):
        if isinstance(document, str):
            text = document
        elif hasattr(document, "read"):
            text = document.read()
        else:
            raise TypeError("document must be text or a readable text stream")

        self._counts = Counter(self._WORD.findall(text.lower()))

    @staticmethod
    def _normalise_word(word: str) -> str:
        if not isinstance(word, str) or not BagOfWords._WORD.fullmatch(word):
            raise ValueError("word must contain one valid word")
        return word.lower()

    def _ordered_items(self) -> list[tuple[str, int]]:
        return sorted(self._counts.items(), key=lambda item: (-item[1], item[0]))

    def __contains__(self, word: object) -> bool:
        return isinstance(word, str) and word.lower() in self._counts

    def __iter__(self):
        return (word for word, _ in self._ordered_items())

    def __str__(self) -> str:
        return ", ".join(f"{word}:{count}" for word, count in self._ordered_items())

    __repr__ = __str__

    def __getitem__(self, word: str) -> int:
        return self._counts[self._normalise_word(word)]

    def __setitem__(self, word: str, count: int) -> None:
        if not isinstance(count, int) or isinstance(count, bool) or count < 0:
            raise ValueError("count must be a non-negative integer")
        word = self._normalise_word(word)
        if count == 0:
            self._counts.pop(word, None)
        else:
            self._counts[word] = count

    def __add__(self, other: object):
        if not isinstance(other, BagOfWords):
            return NotImplemented
        result = BagOfWords("")
        result._counts = self._counts + other._counts
        return result


# Self-contained checks.
bow1 = BagOfWords("ala ma kota ala ma ala")
bow2 = BagOfWords("tomek tez ma kota")
bow3 = bow1 + bow2
assert str(bow1) == "ala:3, ma:2, kota:1"
assert list(bow1) == ["ala", "ma", "kota"]
assert "tomek" not in bow1 and "tomek" in bow3 and "ala" in bow3
assert str(bow3) == "ala:3, ma:3, kota:2, tez:1, tomek:1"
assert bow1["ala"] == 3 and bow3["ala"] == 3
bow3["tomek"] = 10
assert next(iter(bow3)) == "tomek"
print("BagOfWords checks: OK")

BagOfWords checks: OK
